# Cleaning Checks 
In order to effectively run a model, each column within the data soruces needs to be properly standarized and non-redundant. The checks are as followed:

1. **Remove Duplicates First, Then Drop Unwanted Columns**
2. **Strip Whitespace and Handle Blanks**
3. **Make Variants one Value**
4. **Convert Data Types**
5. **Fill Null Values with Past Information**
6. **Split into tables and then Validate**
7. **Load New Files to Processed Folder**

In [1]:
# Load in Data and Libraries
import pandas as pd
import numpy as np
from pathlib import Path

# The notebook lives in notebooks/, so the project root is one folder up
PROJECT_ROOT  = Path.cwd().parent
RAW_DATA      = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

tix = pd.read_csv(RAW_DATA / "ticket_activity_2025_26.csv", dtype=str, keep_default_na=False)
ren = pd.read_csv(RAW_DATA / "renewals_2026_27.csv", dtype=str, keep_default_na=False)

In [2]:
# Step 1: Remove Duplicates, then Drop Unwanted Columns

tix = tix.drop_duplicates()
tix = tix.drop(columns = ["Unnamed: 23", "Notes", "Email", "Name"])

tix.head()

,Account_ID,Age,Location,Account_Type,Member_Since,Event_Name,Event_Type,Event_Date,Section,Row,...,Num_Seats,Price_Code,Ticket_Type,Price_Per_Seat,Payment_Plan,Payment_Status,Seat_Status,Tickets_Scanned,Resale_Price,Rep_Code
0,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 01 vs Magic,Game,2025-11-01,109,I,...,4,M_F,Full Season,$405.00,Pay in Full,GREEN,,3,,jm
1,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 02 vs Bulls,Game,2025-11-02,109,I,...,4,M_F,Full Season,405.0,Pay in Full,Green,,3,,
2,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 03 vs Mavericks,Game,11/04/2025,109,I,...,4,M_F,Full Season,405.0,Pay in Full,Green,,3,,
3,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 04 vs Jazz,Game,2025-11-06,109,I,...,,M_F,Full Season,$405.00,Pay in Full,Green,,4,,jm
4,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 05 vs Clippers,Game,11/10/2025,109,I,...,4,M_F,Full Season,405.0,,Green,,,,


In [3]:
# Step 2: Strip Whitespace and Handle Blanks 

tix = tix.apply(lambda c: c.str.strip())

# Seat_Status blanks MEAN something, so label them before blanks become NaN
tix["Seat_Status"] = tix["Seat_Status"].str.upper().replace({"": "KEPT", "N/A": "KEPT", "-": "KEPT"})

# Every other fake blank becomes real missing
tix = tix.replace({"": np.nan, "N/A": np.nan, "n/a": np.nan, "-": np.nan, "unknown": np.nan})

tix.head(25)

,Account_ID,Age,Location,Account_Type,Member_Since,Event_Name,Event_Type,Event_Date,Section,Row,...,Num_Seats,Price_Code,Ticket_Type,Price_Per_Seat,Payment_Plan,Payment_Status,Seat_Status,Tickets_Scanned,Resale_Price,Rep_Code
0,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 01 vs Magic,Game,2025-11-01,109,I,...,4,M_F,Full Season,$405.00,Pay in Full,GREEN,KEPT,3,NaN,jm
1,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 02 vs Bulls,Game,2025-11-02,109,I,...,4,M_F,Full Season,405.0,Pay in Full,Green,KEPT,3,NaN,NaN
2,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 03 vs Mavericks,Game,11/04/2025,109,I,...,4,M_F,Full Season,405.0,Pay in Full,Green,KEPT,3,NaN,NaN
3,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 04 vs Jazz,Game,2025-11-06,109,I,...,NaN,M_F,Full Season,$405.00,Pay in Full,Green,KEPT,4,NaN,jm
4,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 05 vs Clippers,Game,11/10/2025,109,I,...,4,M_F,Full Season,405.0,NaN,Green,KEPT,NaN,NaN,NaN
5,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 06 vs 76ers,Game,2025-11-19,109,I,...,4,M_F,Full Season,405.0,Pay in Full,Green,KEPT,4,NaN,NaN
6,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 07 vs Hawks,Game,11/28/2025,109,I,...,4,M_F,Full Season,405.0,PIF,Green,KEPT,2,NaN,NaN
7,ACCT100000,31,"New York, NY",Individual,2024,NYK HG08 vs Suns,Game,2025-12-08,109,I,...,4,M_F,Full Season,$405.00,Pay in Full,Green,KEPT,3,NaN,KT
8,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 09 vs Mavericks,Game,12/18/2025,109,I,...,4,M_F,Full Season,$405.00,Pay in Full,Green,KEPT,4,NaN,NaN
9,ACCT100000,31,"New York, NY",Individual,2024,Knicks Home Game 10 vs Magic,Game,2025-12-20,109,I,...,4,M_F,NaN,405.0,Pay in Full,Green,KEPT,3,NaN,NaN


In [4]:
# Step 3: Map Variants to One Value within Ticekting Data Set

maps = {
   "Payment_Plan": {
    "pay in full": "Pay in Full", "pif": "Pay in Full",
    "3 way payment plan": "3 Way", "3-pay": "3 Way", "3 way pmt plan": "3 Way", "3 way": "3 Way",
    "5-6 month payment plan": "5-6 Month", "5-6 mo plan": "5-6 Month", "6 month plan": "5-6 Month", "5-6 month": "5-6 Month",
    "12 payments, monthly": "12 Monthly", "12 monthly": "12 Monthly", "monthly (12)": "12 Monthly",
},
    "Payment_Status": {
        "green": "Green", "grn": "Green",
        "yellow": "Yellow", "yel": "Yellow",
        "red": "Red",
    },
    "Account_Type": {"individual": "Individual", "corporate": "Corporate"},
    "Event_Type":   {"game": "Game", "concert": "Concert"},
}

for col, mapping in maps.items():
    tix[col] = tix[col].str.strip().str.lower().map(mapping)

for col in maps:
    print(col, tix[col].isna().sum())

Payment_Plan 762
Payment_Status 455
Account_Type 1477
Event_Type 744


In [5]:
# Step 4: Convert Types for Numeric Columns
tix["Age"] = pd.to_numeric(tix["Age"].str.extract(r"(-?\d+)")[0], errors="coerce")
tix.loc[(tix["Age"] < 18) | (tix["Age"] > 100), "Age"] = np.nan

for col in ["Price_Per_Seat", "Resale_Price"]:
    tix[col] = pd.to_numeric(tix[col].str.replace("$", "", regex=False), errors="coerce")
for col in ["Num_Seats", "Tickets_Scanned", "Member_Since"]:
    tix[col] = pd.to_numeric(tix[col], errors="coerce")

tix["Event_Date"] = pd.to_datetime(tix["Event_Date"], format="mixed", errors="coerce")


tix.dtypes

Account_ID                    str
Age                       float64
Location                      str
Account_Type                  str
Member_Since              float64
Event_Name                    str
Event_Type                    str
Event_Date         datetime64[us]
Section                       str
Row                           str
Seat_Numbers                  str
Num_Seats                 float64
Price_Code                    str
Ticket_Type                   str
Price_Per_Seat            float64
Payment_Plan                  str
Payment_Status                str
Seat_Status                   str
Tickets_Scanned           float64
Resale_Price              float64
Rep_Code                      str
dtype: object

In [6]:
# Step 5a: Clean/Split Price Code into Price Level, Plan Codes and Seating Area. Then Maps Ticket Types, and Event Types through the use of the new columns.


tix["Price_Code"] = (tix["Price_Code"]
                     .str.strip()
                     .str.upper()                              # "c_f" -> "C_F"
                     .str.replace("-", "_", regex=False))      # "C-F" -> "C_F"



tix["Event_Name"] = (tix["Event_Name"]
                     .str.strip()
                     .str.upper()
                     .str.replace("NYK HG", "KNICKS HOME GAME ", regex=False))

print(tix["Event_Name"].nunique())   # 49






tix["Price_Level"] = tix["Price_Code"].str.split("_").str[0] # Splits the Price Code and takes the first Part as the Price Level. "C" or "22"
tix["Plan_Code"] = tix["Price_Code"].str.split("_").str[-1] # Splits the Price Code and takes the second Part as the Plan Code. Full, half or Quarter Seasons
tix["Seating_Area"] = np.where(tix["Price_Level"].str.isdigit(), "Balcony", "Lower") # If the Price Level is a number, then the Seating Area is Balcony, otherwise it is Lower

# Mapping Ticket Type and Event Type on Plan Code and Event Name.
tix["Ticket_Type"] = tix["Plan_Code"].map({"F": "Full Season", "H": "Half Season", "Q": "Quarter Season"}) # Maps the Plan Code to Ticket Type
tix["Event_Type"] = np.where(tix["Event_Name"].str.startswith("CONCERT"), "Concert", "Game") # If the Event Name starts with Concert, then the Event Type is Concert, otherwise it is Game

# Split Seat Numbers into Start and end, change to numeric, then calculate number of seats. 
ends = tix["Seat_Numbers"].str.split("-", expand = True).apply(pd.to_numeric, errors = "coerce")
tix["Num_Seats"] = ends[1].fillna(ends[0]) - ends[0] + 1 # If the Seat Numbers are in the format "1-4", then the Num Seats is 4 - 1 + 1 = 4. If the Seat Numbers are in the format "1", then the Num Seats is 1 - 1 + 1 = 1.

for col in ["Price_Level", "Plan_Code", "Seating_Area"]:
    tix[col] = tix[col].str.strip().str.upper()

tix.head(25)





49


,Account_ID,Age,Location,Account_Type,Member_Since,Event_Name,Event_Type,Event_Date,Section,Row,...,Price_Per_Seat,Payment_Plan,Payment_Status,Seat_Status,Tickets_Scanned,Resale_Price,Rep_Code,Price_Level,Plan_Code,Seating_Area
0,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 01 VS MAGIC,Game,2025-11-01,109,I,...,405.0,Pay in Full,Green,KEPT,3.0,NaN,jm,M,F,LOWER
1,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 02 VS BULLS,Game,2025-11-02,109,I,...,405.0,Pay in Full,Green,KEPT,3.0,NaN,NaN,M,F,LOWER
2,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 03 VS MAVERICKS,Game,2025-11-04,109,I,...,405.0,Pay in Full,Green,KEPT,3.0,NaN,NaN,M,F,LOWER
3,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 04 VS JAZZ,Game,2025-11-06,109,I,...,405.0,Pay in Full,Green,KEPT,4.0,NaN,jm,M,F,LOWER
4,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 05 VS CLIPPERS,Game,2025-11-10,109,I,...,405.0,NaN,Green,KEPT,NaN,NaN,NaN,M,F,LOWER
5,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 06 VS 76ERS,Game,2025-11-19,109,I,...,405.0,Pay in Full,Green,KEPT,4.0,NaN,NaN,M,F,LOWER
6,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 07 VS HAWKS,Game,2025-11-28,109,I,...,405.0,Pay in Full,Green,KEPT,2.0,NaN,NaN,M,F,LOWER
7,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 08 VS SUNS,Game,2025-12-08,109,I,...,405.0,Pay in Full,Green,KEPT,3.0,NaN,KT,M,F,LOWER
8,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 09 VS MAVERICKS,Game,2025-12-18,109,I,...,405.0,Pay in Full,Green,KEPT,4.0,NaN,NaN,M,F,LOWER
9,ACCT100000,31.0,"New York, NY",Individual,2024.0,KNICKS HOME GAME 10 VS MAGIC,Game,2025-12-20,109,I,...,405.0,Pay in Full,Green,KEPT,3.0,NaN,NaN,M,F,LOWER


In [7]:
# Step 5B: Check for Conflicts in Account Info. All columns related to Account Info should have the Same Value for Each Account ID. If there are any conflicts, then there is an issue with the data. 

ACCOUNT_COLS = ["Age", "Location", "Account_Type", "Member_Since"] # Columns related to the account information
PLAN_COLS = ["Ticket_Type", "Price_Code", "Price_Level", "Price_Per_Seat", "Payment_Plan", "Payment_Status", "Section", "Row", "Seat_Numbers", "Num_Seats", "Seating_Area", "Plan_Code"] # Columns related to the plan information

conflicts = tix.groupby("Account_ID")[ACCOUNT_COLS + PLAN_COLS].nunique().gt(1).sum() # Grops by accoutn ID, and counts num of values that are unique in each column. If values greater than one, then there is a conflict within data. 
conflicts[conflicts > 0] # Shows the columns that have conflicts within the data.

Price_Per_Seat    290
dtype: int64

In [8]:
# Step 5C: Resolve to One Value per Account and Per Event

def most_common_value(values_for_one_account):
    """
    Given one account's values for one column, return the value that appears most frequently. Blanks (NaN) are ignored. If every value is blank, return NaN
    """
    counts = values_for_one_account.value_counts()  # tally the number of occurences of each value, ignoring NaN
    if counts.empty:
        return np.nan
    return counts.index[0] # return the value that appears most frequently

# Built One Summary Row Per Account
account_columns = ACCOUNT_COLS + PLAN_COLS # Every Column relating to Plan Info and Account Info
grouped_by_account = tix.groupby("Account_ID") # Split 917 Accounts
account_profiles = grouped_by_account[account_columns].agg(most_common_value) # For Each Account, take the most common value for each column. This will resolve any coinflicts within the data.


example_account_profile = account_profiles.loc["ACCT100000"]
print(example_account_profile) # Shows the most common value for each column for account ACCT100000. This is the profile for that account.

#Events Section: Built One Summary Row Per Event
grouped_by_event = tix.groupby("Event_Name") # Split by Games or Concerts
event_details = grouped_by_event[["Event_Type", "Event_Date"]].agg(most_common_value) # For Each Event, take the most common value for each column. This will resolve any coinflicts within the data.



Age                       31.0
Location          New York, NY
Account_Type        Individual
Member_Since            2024.0
Ticket_Type        Full Season
Price_Code                 M_F
Price_Level                  M
Price_Per_Seat           405.0
Payment_Plan       Pay in Full
Payment_Status           Green
Section                    109
Row                          I
Seat_Numbers              8-11
Num_Seats                  4.0
Seating_Area             LOWER
Plan_Code                    F
Name: ACCT100000, dtype: object


In [9]:
# Step 5C Continued: Backup Fill For Prices
# If an account price was blank on every card, use the typical price for its price code instead. 

# 1. Typical (median) price for each price code, e.g. "M_F" -> 405.0
median_price_by_code = account_profiles.groupby("Price_Code")["Price_Per_Seat"].median()

# 2. Look up each account's backup price based on its price code
backup_price = account_profiles["Price_Code"].map(median_price_by_code)

# 3. Use the backup ONLY where the real price is missing
account_profiles["Price_Per_Seat"] = account_profiles["Price_Per_Seat"].fillna(backup_price)

In [10]:
# Step 5c: What still is Missing

still_missing = account_profiles.isna().sum()
still_missing[still_missing > 0]

Age         79
Location    71
dtype: int64

In [11]:
# Build New Tables and Test 

# Account-level tables: one row per account (from Step 5c's summary cards) ----
accounts = account_profiles[ACCOUNT_COLS].reset_index()   # who the member is
plans    = account_profiles[PLAN_COLS].reset_index()      # what they bought and how they pay

# Event-level table: one row per game or concert ----
events = event_details.reset_index()

# Attendance: one row per account x event (the only columns that change game to game) ----
event_level_columns = ["Account_ID", "Event_Name", "Seat_Status", "Tickets_Scanned", "Resale_Price"]
attendance = tix[event_level_columns].reset_index(drop=True)

# Renewals: one row per account, target = 1 renewed / 0 cancelled
ren["Account_ID"] = ren["Account_ID"].str.strip().str.upper()           # join check fix: 14 lowercase IDs
renewal_map = {"renewed": 1, "y": 1, "yes": 1, "cancelled": 0, "canceled": 0, "n": 0, "no": 0}
ren["renewed"] = ren["Renewal_Status"].str.strip().str.lower().map(renewal_map)
renewals = ren[["Account_ID", "renewed"]]                               # leakage columns left out on purpose


In [12]:
# Look At Tables

tables = {"accounts": accounts, "plans": plans, "events": events,
          "attendance": attendance, "renewals": renewals}

for name, table in tables.items():
    print(f"{name:<11} {table.shape[0]:>6,} rows  x {table.shape[1]:>2} columns")

accounts       917 rows  x  5 columns
plans          917 rows  x 13 columns
events          49 rows  x  3 columns
attendance  24,752 rows  x  5 columns
renewals       889 rows  x  2 columns


In [13]:
# Prove Tables are Correct and Right

# 1. GRAIN: one row per thing the table describes
assert len(accounts) == 917, "accounts should have 917 rows"
assert accounts["Account_ID"].is_unique, "an account appears twice in accounts"
assert len(plans) == 917 and plans["Account_ID"].is_unique, "plans should have one row per account"
assert len(events) == 49, f"expected 49 events, got {len(events)}"
assert events["Event_Name"].is_unique, "an event appears twice in events"
assert not attendance.duplicated(["Account_ID", "Event_Name"]).any(), "an account has the same event twice"
assert renewals["Account_ID"].is_unique, "an account appears twice in renewals"

# 2. REFERENTIAL INTEGRITY: every ID points to something that exists
assert attendance["Account_ID"].isin(accounts["Account_ID"]).all(), "attendance has an unknown account"
assert attendance["Event_Name"].isin(events["Event_Name"]).all(), "attendance has an unknown event"
assert renewals["Account_ID"].isin(accounts["Account_ID"]).all(), "renewals has an unknown account"

# 3. ALLOWED VALUES: only clean codes made it through
allowed_seat_codes = {"KEPT", "F", "RE", "O", "D", "P"}
assert set(attendance["Seat_Status"]).issubset(allowed_seat_codes), "unexpected Seat_Status code"
assert renewals["renewed"].notna().all(), "a Renewal_Status spelling was not mapped"

# 4. ROUND TRIP: re-joining everything loses no rows
rebuilt = (attendance
           .merge(accounts, on="Account_ID")
           .merge(plans, on="Account_ID")
           .merge(events, on="Event_Name"))
assert len(rebuilt) == len(attendance), "rows were lost when re-joining the tables"

print("All checks passed.")

All checks passed.


In [14]:
# Seating Area Check

pd.crosstab(plans["Seating_Area"], plans["Section"].str[0])

Section,1,2
Seating_Area,,
BALCONY,0,346
LOWER,571,0


# Cleaning Log

- Split into 5 tables: accounts (917), plans (917), events (49),
  attendance (24,752), renewals (889)
- All validation checks pass: grain, referential integrity, allowed values, round trip
- Seating_Area matches Section for all 917 accounts (571 Bowl, 346 Balcony)
- Renewals: 737 renewed (82.9%), 152 cancelled (17.1%)

In [15]:
# Step 7: Load Tables into New Processed Folder

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)     # creates data/processed/ if it doesn't exist yet

tables = {"accounts": accounts, "plans": plans, "events": events,
          "attendance": attendance, "renewals": renewals}

for name, table in tables.items():
    table.to_csv(PROCESSED_DATA / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(table):,} rows)")

# Business story only, never a model feature (leakage)
ren[["Account_ID", "Cancellation_Reason"]].to_csv(PROCESSED_DATA / "cancellation_reasons.csv", index=False)

Saved accounts.csv (917 rows)
Saved plans.csv (917 rows)
Saved events.csv (49 rows)
Saved attendance.csv (24,752 rows)
Saved renewals.csv (889 rows)
